# Species-Level Diversity, Hands-On

The genus course's notebook (Diversity Metrics, By Hand, in *The Gut
Microbiome, Under Glass*) built Shannon and Simpson diversity from scratch
on genus-level real data. This notebook does the same math, by hand
again, on this course's species-level data — with one important wrinkle
worth catching before the formulas run.

## 1. The wrinkle: these columns don't sum to 100%

The real USA/Malawi dataset's genus columns summed close to 100% per
sample — a full community breakdown. This dataset's 12 species columns
were built to train a classifier (bioinformatics course, notebook 05),
not as a complete community profile — they're a fixed panel of tracked
species, and they sum to very different, much-smaller totals per sample.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../bioinformatics-of-gut-health/toy_species_abundance.csv")
species = [c for c in df.columns if c not in ("sample_id", "health_status")]

row_sums = df[species].sum(axis=1)
row_sums.describe()[["mean", "std", "min", "max"]].round(2)

Nowhere near 100. That means Shannon/Simpson must be computed on
**proportions of this panel** (normalize each row to sum to 1 first),
not on the raw numbers directly the way the genus course's notebook could.
What you'll get is "diversity among these 12 tracked species, relative to
each other" — a real, honest measurement, just of a narrower thing than
"diversity of the whole gut community."

## 2. Shannon diversity, normalized correctly

In [ ]:
def shannon(row):
    vals = row[species].values.astype(float)
    p = vals / vals.sum()      # normalize THIS row's panel to proportions
    p = p[p > 0]
    return -np.sum(p * np.log(p))

df["shannon"] = df.apply(shannon, axis=1)
df.groupby("health_status")["shannon"].describe()[["mean", "std", "min", "max"]].round(3)

### 🔧 YOUR TURN #1
Write `simpson_diversity(row)` yourself, following the genus course
notebook's formula (`1 - sum(p**2)`, using the same normalized `p`), and
apply it the same way `shannon` was applied above.

In [ ]:
# Your code here

## 3. Is the difference real?

In [ ]:
from scipy import stats

healthy = df[df.health_status == "healthy"]["shannon"]
non_healthy = df[df.health_status == "non_healthy"]["shannon"]
t, p = stats.ttest_ind(healthy, non_healthy)
print(f"healthy mean={healthy.mean():.3f}  non_healthy mean={non_healthy.mean():.3f}  p={p:.3f}")

### EXPLAIN #1
*Run the cell above and read the actual p-value — don't guess. The
direction (which group is higher) should match what notebook 03 of the
bioinformatics course would predict. But is this specific result below
the conventional p<0.05 bar? If it's close but not under 0.05, what does
that actually tell you, given everything the stats-focused course (P-values
& False Positives, Multiple-Testing Correction) taught about small,
borderline results?*

> your answer here

## Done — the same honest math, one level down

Same formulas, same by-hand implementation philosophy as the genus
course, applied correctly to data that needed an extra normalization
step first — and a result that's realistically borderline rather than
suspiciously clean, which is exactly what real biological data tends to
look like.

**Next:** `11_species_isnt_the_end_either.ipynb`.